# Sequence Models: RNNs to State-Space Models


RNN figures are from Colah's blog post
*[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)*,
Jurafsky & Martin, *[Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)*
(chapter 8 in the 2025 draft, "RNNs and LSTMs"), and Karpathy's
*[The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/)*.
The state-space diagrams and plots were drawn for this lecture; their scripts
are in the course repository. The gate-by-gate LSTM and GRU material from
earlier years is now the optional module
[M1 LSTMs and GRUs](M1-LSTMs-and-GRUs.qmd).

This lecture uses the book's notation for recurrent networks: $\mathbf{h}_t$
for hidden state, $\boldsymbol{\Omega}$ for weights, $\boldsymbol{\beta}$ for
biases, $\mathrm{a}[\cdot]$ for an activation function. *Understanding Deep
Learning* does not cover state-space models, so the second half uses the
notation of the Mamba papers ($\mathbf{A}, \mathbf{B}, \mathbf{C}, \Delta$),
which is what you will meet when you read them.


## Outline

- Sequences, and three ways to let the past reach the present
- Recurrent neural networks: the recurrence, unrolling, backpropagation through time
- Why gradients vanish or explode: a product of Jacobians
- Why that motivated attention, and what attention costs
- The modern linear-time recurrence: linear attention, state-space models, Mamba, Mamba-2, Gated DeltaNet
- One derivation: the parallel scan
- Why 2026 models alternate recurrent layers with attention layers

# Sequences and three ways to model them

Everything so far has mapped one fixed-size input to one output. Text, audio,
time series, DNA, user sessions and the action stream of an agent are
sequences: variable length, and the order carries the information. This
section sets up the one question every sequence architecture answers
differently, so that RNNs, attention and state-space models can be compared on
the same terms.


## Sequences are everywhere, and order matters

- Text: tokens, $T$ in the thousands to millions
- Speech and audio: 16,000 samples per second
- Time series: sensor readings, prices, patient vitals
- Biology: DNA (3 billion base pairs), protein residues
- Agents: observation, action, observation, action, ...

What they share:

- **Variable length.** A fixed-size input layer does not fit.
- **Order carries meaning.** "dog bites man" is not "man bites dog".
- **The relevant past can be far away.** A pronoun refers back pages.


We want a model whose output at step $t$ depends on
$\mathbf{x}_1, \ldots, \mathbf{x}_t$ (and sometimes on the future as well,
for encoders).

The design question for any sequence model:

**How does step $t$ get to see steps $1, \ldots, t-1$,**
**and what does that cost in time and memory?**


## The autoregressive factorization

Any distribution over a sequence factorizes exactly, by the chain rule of probability:

$$
\mathrm{Pr}(\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_T)
= \prod_{t=1}^{T} \mathrm{Pr}(\mathbf{x}_t \mid \mathbf{x}_1, \ldots, \mathbf{x}_{t-1}).
$$

- So a model of "the next element given the past" is a model of the whole sequence. This is a **language model**, whatever the elements are.
- Training is maximum likelihood (lecture 03): minimize $L = -\sum_t \log \mathrm{Pr}(\mathbf{x}_t \mid \mathbf{x}_{<t}, \boldsymbol{\phi})$, the cross-entropy summed over positions.
- Generation is sampling from each factor in turn, feeding the sample back in.
- The factorization does not say *how* to compute the conditional. That is the architecture, and it is the subject of this lecture and the next.


## Three ways to let the past reach the present

![Three ways the output at step 6 can depend on earlier steps.](figs/09-Sequence-Models/three-ways-to-use-context.png)

**(a) Fixed window / convolution**

Bengio et al. (2003) used the last $n$ words; WaveNet (2016) stacks dilated convolutions.

Context is bounded by design.


**(b) Recurrence**

A state $\mathbf{h}_t$ summarizes everything so far. Unbounded context, constant memory, but the past arrives through $t$ sequential steps.


**(c) Attention** (lecture 10)

Every earlier step is one hop away, weighted by content. Unbounded context, but cost and memory grow with $T$.



## The question behind every sequence model: the state

Every causal sequence model computes $\mathbf{y}_t$ from some **state** that summarizes $\mathbf{x}_{\le t}$. The architectures differ in what that state is:

| | State at step $t$ | Size of state | Cost per new token | Trains in parallel over $t$? |
|---|---|---|---|---|
| Window of $k$ | last $k$ inputs | $O(k)$ | $O(k)$ | yes |
| RNN | one vector $\mathbf{h}_t$ | $O(D)$ | $O(D^2)$ | **no** |
| Attention | all past keys and values | $O(T D)$ | $O(T D)$ | yes |
| Linear recurrence / SSM | a vector or small matrix | $O(D N)$ | $O(D N)$ | **yes** (via a scan) |

Read the table as the plot of the lecture:

- The RNN has the right *inference* profile (constant memory, constant cost per token) but cannot be trained in parallel and forgets.
- Attention trains in parallel and never forgets, but its state grows with the sequence.
- The last row is the 2020s answer: keep the RNN's inference profile, recover parallel training by making the recurrence *linear*, and make it forget *selectively*.


# Recurrent neural networks

The recurrent network is the oldest of the three designs (Elman, 1990), and
the one whose failure mode, vanishing gradients, shaped everything that came
after. We need enough of it to understand that failure precisely, because the
modern recurrences are built to avoid it.


## The recurrence

![A recurrent cell: the hidden state loops back into the cell at the next step. Figure: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).](figs/09-Sequence-Models/rnn-cell-overview.png)


One hidden layer, applied at every step, that also reads its own previous output:

$$
\begin{aligned}
\mathbf{h}_t &= \mathrm{a}\!\left[\boldsymbol{\beta} + \boldsymbol{\Omega}_h \mathbf{h}_{t-1} + \boldsymbol{\Omega}_x \mathbf{x}_t\right], \\
\mathbf{y}_t &= \boldsymbol{\beta}_y + \boldsymbol{\Omega}_y \mathbf{h}_t.
\end{aligned}
$$

- $\mathbf{x}_t \in \mathbb{R}^{D_x}$: input at step $t$ (a token embedding, a frame)
- $\mathbf{h}_t \in \mathbb{R}^{D_h}$: hidden state, $\mathbf{h}_0 = \mathbf{0}$
- $\boldsymbol{\Omega}_h \in \mathbb{R}^{D_h \times D_h}$ recurrent weights, $\boldsymbol{\Omega}_x \in \mathbb{R}^{D_h \times D_x}$ input weights
- $\mathrm{a}[\cdot]$: historically $\tanh$, which keeps $\mathbf{h}_t \in (-1, 1)^{D_h}$
- $\mathbf{y}_t$ feeds a softmax over the vocabulary, or a regression output


Equivalently, one linear layer on the concatenation $[\mathbf{h}_{t-1}; \mathbf{x}_t]$ with weights $[\boldsymbol{\Omega}_h \;|\; \boldsymbol{\Omega}_x]$, which is how most diagrams and most code write it.



## Unrolling in time: one network, shared weights

![The loop (left) is the same thing as the chain (right). Figure: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).](figs/09-Sequence-Models/rnn-unrolled-delay.png)


![The weights $\mathbf{U}, \mathbf{V}, \mathbf{W}$ (our $\boldsymbol{\Omega}_h, \boldsymbol{\Omega}_y, \boldsymbol{\Omega}_x$) are the same at every step. Figure 8.4 of [Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/).](figs/09-Sequence-Models/rnn-unrolled-shared-weights.png)


- Unrolled over $T$ steps, the RNN is a feed-forward network with $T$ layers that **share one set of parameters**. Same idea as a convolution sharing a kernel across positions (lecture 07), here across time.
- The parameter count does not depend on $T$. The depth does.
- Inputs are fed one at a time; $\mathbf{h}_t$ cannot be computed before $\mathbf{h}_{t-1}$.


## Many shapes of sequence problem

![Red: inputs, green: states, blue: outputs. Figure: [Karpathy, 2015](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).](figs/09-Sequence-Models/rnn-configurations.png)

1. One to one: an ordinary feed-forward network
2. One to many: image captioning (one image in, a sentence out)
3. Many to one: sentiment of a review, classification of a time series
4. Many to many, delayed: machine translation (read the source, then write the target)
5. Many to many, synced: next-token prediction, per-frame video labels

The same cell serves all five; what changes is where you read inputs and where you take outputs.

## A character-level language model

![Training on "hello": each input character predicts the next. Figure: [Karpathy, 2015](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).](figs/09-Sequence-Models/char-rnn-hello-example.png)


- Vocabulary $\{\texttt{h}, \texttt{e}, \texttt{l}, \texttt{o}\}$, one-hot inputs of length 4
- Hidden state of length 3 (tiny, for the picture)
- Output: 4 scores, softmax, cross-entropy against the next character
- Note the two `l` inputs: same input, **different outputs** (`l` then `o`). Only the hidden state can tell them apart. That is what the state is for.

Trained on all of Shakespeare, a 3-layer version of this writes plausible blank verse; the samples are in module M1.


## The loss over a sequence

![Teacher forcing: at each step the true previous word is the input and the true next word is the target. Figure 8.6 of [Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/).](figs/09-Sequence-Models/rnn-sequence-loss.png)


$$
\begin{aligned}
L &= \sum_{t=1}^{T} \ell_t, \\
\ell_t &= -\log \mathrm{Pr}(\mathbf{x}_{t+1} \mid \mathbf{h}_t)
\end{aligned}
$$

- $\ell_t$ is the cross-entropy at step $t$ (lecture 03), exactly the autoregressive factorization turned into a loss.
- **Teacher forcing**: during training the input at $t+1$ is the true token, not the model's sample.
- The loss at step $t$ depends on $\boldsymbol{\Omega}_h$ through $\mathbf{h}_t, \mathbf{h}_{t-1}, \ldots, \mathbf{h}_1$. That is the whole difficulty.


## Training: backpropagation through time

- Unroll the network for the sequence (or a chunk of it), run the forward pass, and run ordinary backpropagation (lecture 05) over the unrolled graph. Every step's Jacobians are stored, as for any deep network.
- Because the weights are shared, the gradient with respect to $\boldsymbol{\Omega}_h$ is the **sum** of the gradients from every copy, as in a convolution.
- This is **backpropagation through time** (BPTT), named by Werbos (1990), a direct consequence of the chain rule, not a new algorithm.
- Memory scales with the chunk length $T$, so in practice training uses **truncated BPTT**: run forward over a long sequence, but backpropagate only through the last $k$ steps of each window. Gradients from before the window are simply not computed.
- Training cost is $O(T)$ **sequential** steps per sequence, forward and backward: step $t$ waits for step $t-1$. GPUs that could do the $T$ steps at once sit idle.



## BPTT: the gradient is a sum over time of products of Jacobians

Take one term of the loss, $\ell_t$, and one parameter matrix, $\boldsymbol{\Omega}_h$. By the chain rule along the chain $\mathbf{h}_1 \to \mathbf{h}_2 \to \cdots \to \mathbf{h}_t$:

$$
\frac{\partial \ell_t}{\partial \boldsymbol{\Omega}_h}
= \sum_{k=1}^{t}
\frac{\partial \ell_t}{\partial \mathbf{h}_t}\,
\underbrace{\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_k}}_{\text{the long-range factor}}\,
\frac{\partial^{+} \mathbf{h}_k}{\partial \boldsymbol{\Omega}_h},
\qquad\text{where}\qquad
\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_k}
= \prod_{j=k+1}^{t} \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}.
$$

- The sum over $k$: $\boldsymbol{\Omega}_h$ is used at every step, so each use contributes. $\partial^{+}$ means "the direct contribution at step $k$, holding $\mathbf{h}_{k-1}$ fixed".
- The product: how a change in the state at step $k$ propagates to the state at step $t$, through $t-k$ applications of the same cell.
- The term with $k = t$ is the short-range gradient (one step). The terms with $t - k$ large are what the model needs to *learn* long-range dependencies, and they are a product of $t-k$ Jacobian matrices.
- Everything depends on the size of that product. Next slide: one factor of it.



## One step's Jacobian

Write the pre-activation as $\mathbf{f}_j = \boldsymbol{\beta} + \boldsymbol{\Omega}_h \mathbf{h}_{j-1} + \boldsymbol{\Omega}_x \mathbf{x}_j$, so $\mathbf{h}_j = \mathrm{a}[\mathbf{f}_j]$. Then, exactly as in the backward pass of lecture 05,

$$
\frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}
= \mathrm{diag}\!\left[\mathrm{a}'[\mathbf{f}_j]\right] \boldsymbol{\Omega}_h .
$$

Bound its size with the largest singular value $\sigma_{\max}$ of $\boldsymbol{\Omega}_h$ and a bound $\gamma$ on the activation's slope:

$$
\left\| \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}} \right\|
\le \left\| \mathrm{diag}[\mathrm{a}'[\mathbf{f}_j]] \right\| \cdot \left\| \boldsymbol{\Omega}_h \right\|
\le \gamma\, \sigma_{\max}.
$$

- $\tanh$: $|\tanh'| \le 1$, so $\gamma = 1$
- sigmoid: $|\sigma'| \le 1/4$, so $\gamma = 1/4$


Multiply $t - k$ of them:

$$
\left\| \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_k} \right\|
\le \prod_{j=k+1}^{t} \gamma\,\sigma_{\max}
= \left(\gamma\, \sigma_{\max}\right)^{t-k}.
$$

A geometric sequence in the distance $t-k$. There is no comfortable middle: it goes to $0$ or to $\infty$ unless $\gamma\sigma_{\max}$ is exactly 1.


## Vanishing and exploding gradients

In [ ]:
#| echo: false
#| fig-cap: "The bound $(\\gamma\\sigma_{\\max})^{t-k}$ for five values of $\\gamma\\sigma_{\\max}$, log scale."
#| fig-alt: "Log-scale plot of the gradient bound against steps back in time from 0 to 60. Lines for 0.8 and 0.9 fall to one millionth and one thousandth; the line for 1.0 is flat; lines for 1.1 and 1.25 rise to hundreds and hundreds of thousands."
import numpy as np
import matplotlib.pyplot as plt

factors = {0.8: "#2a78d6", 0.9: "#4a3aa7", 1.0: "#52514e", 1.1: "#eb6834", 1.25: "#e34948"}
k = np.arange(0, 61)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for g, c in factors.items():
    ax.plot(k, g ** k, color=c, lw=2)
    ax.text(k[-1] + 1, g ** k[-1], rf"$\gamma\,\sigma_{{\max}} = {g}$", color=c, va="center", fontsize=10)
ax.set_yscale("log")
ax.set_ylim(1e-7, 1e7)
ax.set_xlim(0, 75)
ax.set_xlabel("steps back in time, $t - k$")
ax.set_ylabel(r"bound on $\|\partial \mathbf{h}_t / \partial \mathbf{h}_k\|$")
ax.axhline(1.0, color="#e4e3df", lw=1, zorder=0)
ax.grid(axis="y", color="#e4e3df", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Pascanu et al. (2013), stated carefully:

- $\sigma_{\max} < 1/\gamma$ is **sufficient** for the long-range gradient to **vanish**: the bound is a decaying geometric series, so gradients from far back contribute nothing and the model cannot learn a dependency it is not already close to representing.
- $\sigma_{\max} > 1/\gamma$ is **necessary** (not sufficient) for gradients to **explode**: a step in parameter space can then overshoot by orders of magnitude.
- Saturation makes vanishing worse: where $\tanh$ is flat, $\mathrm{a}' \approx 0$ regardless of $\boldsymbol{\Omega}_h$.



## Worked example: the scalar RNN

Take $D_h = 1$: $h_t = \tanh(\omega\, h_{t-1} + \omega_x x_t)$. Then $\partial h_j / \partial h_{j-1} = \tanh'(f_j)\,\omega$, and if the unit is not saturated ($\tanh' \approx 1$):

$$
\frac{\partial h_t}{\partial h_k} \approx \omega^{\,t-k}.
$$

**$\omega = 0.9$, 50 steps back:**
$$0.9^{50} = e^{50 \ln 0.9} = e^{-5.27} \approx 0.005$$

The gradient signal from 50 tokens ago is 0.5% of the one-step signal. How far back is it already below 1%?
$$t - k > \frac{\ln 0.01}{\ln 0.9} = 43.7 \;\Rightarrow\; 44 \text{ steps}.$$


**$\omega = 1.1$, 50 steps back:**
$$1.1^{50} = e^{50 \ln 1.1} = e^{4.77} \approx 117.$$

A gradient 117 times larger than the one-step one; at 100 steps, $1.1^{100} \approx 13{,}800$. One such term dominates the sum and the update step is garbage.


The two regimes are a hair apart in $\omega$. In $D_h$ dimensions the same holds along each singular direction of $\boldsymbol{\Omega}_h$, so some directions vanish while others explode in the same network.


The worked numbers, then a real RNN. The cell runs a 16-unit $\tanh$ RNN
forward on a random 60-step sequence, with $\boldsymbol{\Omega}_h = \omega
\mathbf{Q}$ for an orthogonal $\mathbf{Q}$ so that every singular value is
$\omega$, and multiplies the per-step Jacobians
$\mathrm{diag}[\mathrm{a}'[\mathbf{f}_j]]\,\boldsymbol{\Omega}_h$ backwards from
the last step to get $\|\partial \mathbf{h}_T / \partial \mathbf{h}_{T-n}\|$.
With tiny inputs the units stay unsaturated and the measured norm follows the
bound $\omega^{n}$. Set `input_scale = 0.3` and rerun: the units saturate,
$\tanh' \ll 1$, and *both* curves vanish, the $\omega = 1.1$ one included.
Saturation makes vanishing worse, and $\sigma_{\max} > 1$ is necessary, not
sufficient, for explosion.

In [ ]:
#| code-fold: true
import numpy as np
import matplotlib.pyplot as plt

for omega in (0.9, 1.1):
    print(f"omega = {omega}:  omega^50 = {omega**50:.3g},  omega^100 = {omega**100:.3g}")
n_1pct = np.log(0.01) / np.log(0.9)
print(f"0.9^n < 1% once n > ln(0.01)/ln(0.9) = {n_1pct:.1f}, i.e. after {int(np.ceil(n_1pct))} steps\n")

D_h, D_x, T = 16, 4, 60
input_scale = 0.001                        # try 0.3: the units saturate
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.normal(size=(D_h, D_h)))   # orthogonal: every singular value is 1
Omega_x = input_scale * rng.normal(size=(D_h, D_x))
beta = np.zeros(D_h)
x = rng.normal(size=(T, D_x))

def rnn_forward(Omega_h, x):
    h, hs, fs = np.zeros(D_h), [], []
    for x_t in x:
        f = beta + Omega_h @ h + Omega_x @ x_t     # pre-activation f_t
        h = np.tanh(f)
        hs.append(h)
        fs.append(f)
    return np.array(hs), np.array(fs)

def jacobian_norms(Omega_h, fs):
    """||dh_T / dh_{T-n}||_2 for n = 0, 1, ..., T-1."""
    J, norms = np.eye(D_h), [1.0]
    for f in fs[:0:-1]:                            # f_T, f_{T-1}, ..., f_2
        J = J @ np.diag(1 - np.tanh(f) ** 2) @ Omega_h
        norms.append(np.linalg.norm(J, 2))
    return np.array(norms)

n = np.arange(T)
fig, ax = plt.subplots(figsize=(7, 3.8))
for omega, c in [(0.9, "#4a3aa7"), (1.1, "#eb6834")]:
    hs, fs = rnn_forward(omega * Q, x)
    norms = jacobian_norms(omega * Q, fs)
    print(f"sigma_max = {omega}: ||dh_T/dh_(T-50)|| = {norms[50]:.3g} (bound {omega**50:.3g}), "
          f"max |h_t| = {np.abs(hs).max():.2f}")
    ax.plot(n, norms, color=c, lw=2, label=rf"measured, $\sigma_{{\max}} = {omega}$")
    ax.plot(n, omega ** n, color=c, lw=1, ls="--", label=rf"bound ${omega}^{{\,t-k}}$")
ax.set_yscale("log")
ax.set_xlabel("steps back in time, $t - k$")
ax.set_ylabel(r"$\|\partial \mathbf{h}_t / \partial \mathbf{h}_k\|$")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## What it means for learning

![Short gap: $x_0, x_1$ influence $h_3$ through two steps. Figure: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).](figs/09-Sequence-Models/rnn-short-range-dependency.png)


![Long gap: the same inputs must reach $h_{t+1}$ through $t$ steps. Figure: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).](figs/09-Sequence-Models/rnn-long-range-dependency.png)


- A plain RNN learns the short dependency and, in practice, not the long one: the long-range terms of the gradient sum are $10^{-3}$ of the short-range ones, so the optimizer never sees them.
- This is a statement about **training**, not about capacity. The unrolled network could represent the dependency; gradient descent cannot find it.
- Same mathematics as a very deep feed-forward network without residual connections (lecture 08), with the extra constraint that every layer is the *same* layer.

## The fixes: clipping and gates

![An LSTM cell. The top horizontal line, the cell state, passes through only a multiply and an add. Figure: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).](figs/09-Sequence-Models/lstm-repeating-module.png)


**Exploding:** gradient clipping. Rescale the gradient vector when its norm exceeds a threshold (Pascanu et al., 2013). Standard in every LLM recipe today.

**Vanishing:** change the cell so that the state has an **additive** path. The LSTM (Hochreiter & Schmidhuber, 1997) keeps a cell state $\mathbf{c}_t$ updated by

$$
\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t,
$$

with gates $\mathbf{f}_t, \mathbf{i}_t \in (0,1)^{D_h}$ computed from $\mathbf{h}_{t-1}, \mathbf{x}_t$.


Along the cell state, $\partial \mathbf{c}_t / \partial \mathbf{c}_{t-1} \approx \mathrm{diag}[\mathbf{f}_t]$: no weight matrix, no $\tanh$. With $\mathbf{f}_t \approx 1$ the product of Jacobians stays near the identity for as long as the gate says so: the "constant error carousel". The GRU (Cho et al., 2014) does the same with two gates. Gate equations, figures and code are in **module M1**. Keep the idea: **a learned, input-dependent forget factor on an additive state.** It returns in Mamba.



## Where RNNs went: seq2seq, the bottleneck, attention

- With LSTMs, recurrent models ran speech recognition, translation and captioning from about 2013 to 2017.
- **Sequence to sequence** (Sutskever, Vinyals & Le, 2014): an encoder LSTM reads the source sentence into one final state; a decoder LSTM generates the target from it. Every fact about a 40-word sentence has to survive in one vector: the **encoder bottleneck**.
- **Attention** (Bahdanau, Cho & Bengio, 2014): let the decoder look back at *all* encoder states, weighted by relevance, at every output step. The bottleneck disappears; the recurrence stays.
- **Attention is all you need** (Vaswani et al., 2017): remove the recurrence entirely. Self-attention lets every position read every other in one step, and the whole sequence is processed at once.
- Lecture 10 covers attention and the transformer in full. Here we need only *why* it won, in two numbers.



## Why attention won: path length and parallelism

Table 1 of Vaswani et al. (2017), for sequence length $T$, width $D$, kernel size $k$:

| Layer type | Compute per layer | Sequential operations | Maximum path length |
|---|---|---|---|
| Recurrent | $O(T D^2)$ | $O(T)$ | $O(T)$ |
| Convolutional | $O(k\, T D^2)$ | $O(1)$ | $O(\log_k T)$ |
| Self-attention | $O(T^2 D)$ | $O(1)$ | $O(1)$ |

- **Path length** is the number of steps a gradient must travel between two positions. For the RNN it is $t - k$, and we have just seen what a product of $t-k$ Jacobians does. For attention it is 1: **no product, no vanishing**, at any distance.
- **Sequential operations** is how many steps cannot be parallelized. The RNN's $O(T)$ means training time grows with sequence length no matter how many GPUs you have. Attention's $O(1)$ means a 4,096-token sequence is one big matrix multiply.
- These two properties, not raw accuracy on 2017 benchmarks, are why the transformer scaled to GPT-3 and the LSTM did not.


## What attention pays for it

**Training:** compute per layer is $O(T^2 D)$. Doubling the context quadruples the attention cost. Manageable up to tens of thousands of tokens with good kernels.

**Inference:** to generate token $T+1$, the model must read the keys and values of all $T$ previous tokens. They are kept in a **KV cache**, which costs

$$
\text{memory} = T \times L \times 2 \times H_{kv} \times d_{\text{head}} \times \text{bytes}
$$

($L$ layers, $H_{kv}$ key-value heads) and grows **linearly with $T$**, per sequence, for the whole conversation.

Symbols: this lecture writes $T$ for sequence length and $H_{kv}$ for KV heads; lectures 10 and 11 write $N$ and $n_{kv}$.


Compare the RNN at inference: one state vector, $O(1)$ memory and $O(1)$ compute per token, regardless of how long the conversation has been.

So the question that drives the second half of this lecture:

**Can we build a recurrence that**

1. trains in parallel like attention,
2. does not forget by accident like the RNN,
3. keeps the RNN's constant-size state at inference?

The answer since about 2023 is yes, with conditions, and the conditions are the interesting part.


# The modern linear-time recurrence

The transformer's weakness is the thing the RNN did well: constant cost per
token at inference. From 2020 on, several lines of work converged on the same
object, a recurrence that is *linear* in the state, with coefficients that
depend on the input. Linearity is what makes it trainable in parallel; the
input dependence is what lets it forget on purpose rather than by accident.
This section builds that object three times, from attention, from control
theory, and from online learning, and shows they are one family.

A notation note: from here on we use the symbols of the Mamba papers
($\mathbf{A}, \mathbf{B}, \mathbf{C}$ for the recurrence, $\Delta$ for the
step size, $N$ for the state dimension per channel), because that is what the
papers and the code use. The book does not cover this material.


## Linear attention: drop the softmax

Causal softmax attention (lecture 10) at position $t$, with queries, keys and values $\mathbf{q}_s, \mathbf{k}_s, \mathbf{v}_s$:

$$
\mathbf{y}_t = \sum_{s \le t} \frac{\exp(\mathbf{q}_t^{\mathsf T} \mathbf{k}_s)}{\sum_{r \le t} \exp(\mathbf{q}_t^{\mathsf T} \mathbf{k}_r)}\, \mathbf{v}_s .
$$

Katharopoulos et al. (2020): replace $\exp(\mathbf{q}^{\mathsf T}\mathbf{k})$ by any kernel that factorizes, $\phi(\mathbf{q})^{\mathsf T}\phi(\mathbf{k})$ with a feature map $\phi$ (they used $\mathrm{elu}(\cdot) + 1$). Then the sum over $s$ no longer depends on $\mathbf{q}_t$ and can be pulled out:

$$
\mathbf{y}_t
= \frac{\sum_{s \le t} \mathbf{v}_s\, \phi(\mathbf{k}_s)^{\mathsf T}\, \phi(\mathbf{q}_t)}{\sum_{s \le t} \phi(\mathbf{k}_s)^{\mathsf T} \phi(\mathbf{q}_t)}
= \frac{\mathbf{S}_t\, \phi(\mathbf{q}_t)}{\mathbf{z}_t^{\mathsf T}\, \phi(\mathbf{q}_t)},
$$

$$
\text{where}\qquad
\mathbf{S}_t = \sum_{s \le t} \mathbf{v}_s \phi(\mathbf{k}_s)^{\mathsf T}
\quad\text{and}\quad
\mathbf{z}_t = \sum_{s \le t} \phi(\mathbf{k}_s).
$$



## Linear attention is an RNN with a matrix state

The two sums are running totals, so they obey a recurrence:

$$
\begin{aligned}
\mathbf{S}_t &= \mathbf{S}_{t-1} + \mathbf{v}_t\, \phi(\mathbf{k}_t)^{\mathsf T}, \\
\mathbf{y}_t &= \mathbf{S}_t\, \phi(\mathbf{q}_t)\ (\text{normalized}).
\end{aligned}
$$

- The state $\mathbf{S}_t$ is a $d_v \times d_k$ **matrix**, not a vector: an associative memory that stores key-value pairs as a sum of outer products.
- Reading it with a query returns $\sum_s \mathbf{v}_s\, (\phi(\mathbf{k}_s)^{\mathsf T}\phi(\mathbf{q}_t))$: each stored value weighted by its key's similarity to the query.
- Per token: $O(d_k d_v)$ compute and memory, independent of $T$. The title of the paper is the whole point: *Transformers are RNNs*.


**Two catches**, which the rest of the section fixes:

1. **No forgetting.** $\mathbf{S}_t$ only ever adds. After $T \gg d_k$ tokens the outer products interfere and old associations blur. The RNN forgot by accident; this one cannot forget at all.
2. **Quality.** Without the softmax's sharp selection, linear attention trails softmax attention on language modelling at equal size. The feature map $\phi$ is a weak substitute for $\exp$.



## The general form: a linear recurrence with data-dependent coefficients

Every model in the rest of this lecture is a special case of

$$
\boxed{\;\mathbf{h}_t = \mathbf{A}_t\, \mathbf{h}_{t-1} + \mathbf{B}_t\, \mathbf{x}_t,
\qquad
\mathbf{y}_t = \mathbf{C}_t\, \mathbf{h}_t\;}
$$

a recurrence that is **linear in the state** $\mathbf{h}$, with coefficients $(\mathbf{A}_t, \mathbf{B}_t, \mathbf{C}_t)$ that may be constants or functions of the current input $\mathbf{x}_t$.

| Model | $\mathbf{A}_t$ | Input-dependent? |
|---|---|---|
| Linear attention (2020) | $\mathbf{I}$ | no |
| S4 (2022) | $\exp(\Delta\mathbf{A})$, fixed | no |
| RetNet (2023) | $\gamma\,\mathbf{I}$, fixed decay | no |
| Mamba (2023) | $\exp(\Delta_t \mathbf{A})$, diagonal | yes |
| Mamba-2 (2024) | $a_t\,\mathbf{I}$, one scalar per head | yes |
| Gated DeltaNet (2024) | $\alpha_t(\mathbf{I} - \beta_t \mathbf{k}_t\mathbf{k}_t^{\mathsf T})$ | yes |


Compared with the RNN, $\mathbf{h}_t = \mathrm{a}[\boldsymbol{\Omega}_h \mathbf{h}_{t-1} + \ldots]$:

- **No nonlinearity between steps.** The Jacobian $\partial \mathbf{h}_t / \partial \mathbf{h}_{t-1} = \mathbf{A}_t$ is chosen, not learned through a $\tanh$, so its size is under control: $\|\mathbf{A}_t\| \le 1$ by construction.
- **Linearity is what buys parallel training.** $\mathbf{h}_t$ is a linear function of all earlier inputs; the whole sequence can be computed at once by a scan (derived below) or as a matrix.
- Nonlinearity lives *between* layers (gates, MLPs), as in a transformer.


## State-space models: start in continuous time

Control theory's linear time-invariant system, for a one-dimensional signal $x(t)$ (one channel):

$$
\frac{d\mathbf{h}(t)}{dt} = \mathbf{A}\,\mathbf{h}(t) + \mathbf{B}\,x(t),
\qquad
y(t) = \mathbf{C}\,\mathbf{h}(t),
$$

with $\mathbf{h}(t) \in \mathbb{R}^{N}$, $\mathbf{A} \in \mathbb{R}^{N \times N}$, $\mathbf{B} \in \mathbb{R}^{N \times 1}$, $\mathbf{C} \in \mathbb{R}^{1 \times N}$.

- $\mathbf{h}$ is an $N$-dimensional latent state compressing the history of a scalar input; $\mathbf{A}$ says how it evolves on its own, $\mathbf{B}$ how input enters, $\mathbf{C}$ what we read out.
- Gu et al. (2020, 2021): choose $\mathbf{A}$ so that $\mathbf{h}(t)$ is the coefficients of a **polynomial approximation of the input's history** (HiPPO, Legendre basis). Then the state is an optimal fixed-size memory of everything since $t = 0$, by construction, not by learning.
- A deep learning layer applies one such system to **each of the $D$ channels** of the input independently ($D$ separate SSMs with their own $\mathbf{B}, \mathbf{C}$, sharing or not sharing $\mathbf{A}$) and mixes channels with a linear layer afterwards.
- To use it on tokens we need a discrete-time version.



## Discretization: from differential equation to recurrence

Sample every $\Delta$ units of time and hold the input constant in between (**zero-order hold**). The ODE then has an exact solution per step:

$$
\mathbf{h}_t = \bar{\mathbf{A}}\, \mathbf{h}_{t-1} + \bar{\mathbf{B}}\, x_t,
\qquad
\bar{\mathbf{A}} = \exp(\Delta \mathbf{A}),
\qquad
\bar{\mathbf{B}} = (\Delta\mathbf{A})^{-1}\left(\exp(\Delta\mathbf{A}) - \mathbf{I}\right)\Delta\mathbf{B}.
$$

The cheap alternative is Euler's method: $\bar{\mathbf{A}} \approx \mathbf{I} + \Delta\mathbf{A}$, $\bar{\mathbf{B}} \approx \Delta \mathbf{B}$.

**Worked number** (one state, $N=1$): $A = -1$, $B = 1$, $\Delta = 0.1$.

- ZOH: $\bar a = e^{-0.1} = 0.905$, $\;\bar b = \frac{e^{-0.1} - 1}{-0.1} \cdot 0.1 = 1 - e^{-0.1} = 0.095$
- Euler: $\bar a \approx 1 - 0.1 = 0.9$, $\;\bar b \approx 0.1$
- The recurrence $h_t = 0.905\, h_{t-1} + 0.095\, x_t$ is an exponential moving average with a memory of about $1/\Delta = 10$ steps.


What the pieces mean:

- A negative real part of $\mathbf{A}$ (always, by initialization) gives $|\bar{\mathbf{A}}| < 1$: the state decays, never explodes. The vanishing-gradient bound of the RNN is now a *design parameter*.
- $\Delta$ is the **time scale**: how long one token lasts. Small $\Delta$, the state barely changes; large $\Delta$, the state is mostly overwritten by the current input.


The worked number in code, and why zero-order hold rather than Euler. Fed a
constant input, the ZOH recurrence lands exactly on the ODE's solution
$1 - e^{-t}$ at every sample, whatever $\Delta$; Euler does not. The table
shows the larger failure: for $\Delta > 1$ Euler's $\bar a = 1 + \Delta A$ goes
negative (the state flips sign every step), and for $\Delta > 2$ it exceeds 1 in
magnitude and the recurrence explodes, while ZOH's $\bar a = e^{\Delta A}$ stays
in $(0, 1)$ for every $\Delta$. A model that *learns* $\Delta$ needs that
guarantee. Change `A` or `Delta` and rerun.

In [ ]:
import numpy as np

A, B = -1.0, 1.0      # continuous-time SSM, one state (N = 1)
Delta = 0.1           # step size

a_zoh = np.exp(Delta * A)
b_zoh = (np.exp(Delta * A) - 1) / (Delta * A) * Delta * B
a_euler, b_euler = 1 + Delta * A, Delta * B
print(f"Delta = {Delta}: ZOH a_bar = {a_zoh:.3f}, b_bar = {b_zoh:.3f};  "
      f"Euler a_bar = {a_euler:.3f}, b_bar = {b_euler:.3f}")
print(f"memory: a_bar^k = exp(k Delta A) falls to 1/e after k = 1/(Delta |A|) = {1 / (Delta * abs(A)):.0f} steps")

t_end = 1.0           # feed x = 1 from time 0; the exact solution is (B/-A)(1 - exp(A t))
h_zoh = h_euler = 0.0
for _ in range(round(t_end / Delta)):
    h_zoh = a_zoh * h_zoh + b_zoh * 1.0
    h_euler = a_euler * h_euler + b_euler * 1.0
print(f"\nstep input, h at t = {t_end}: exact {B / -A * (1 - np.exp(A * t_end)):.4f}, "
      f"ZOH {h_zoh:.4f}, Euler {h_euler:.4f}")

print(f"\n{'Delta':>6} {'ZOH a_bar':>10} {'ZOH b_bar':>10} {'Euler a_bar':>12} {'Euler b_bar':>12}")
for d in [0.05, 0.1, 0.3, 1.0, 1.5, 2.5]:
    a = np.exp(d * A)
    print(f"{d:>6} {a:>10.3f} {(a - 1) / (d * A) * d * B:>10.3f} {1 + d * A:>12.3f} {d * B:>12.3f}")

## $\Delta$ is a forget knob

![How much of an input survives in the state after $k$ further steps, for three step sizes, with $A = -1$.](figs/09-Sequence-Models/discretization-step.png)


With $A = -1$, the input that arrived $k$ steps ago is weighted $\bar a^{\,k} = e^{-k\Delta}$ in the current state.

- $\Delta = 0.05$: remembers for about $1/\Delta = 20$ steps
- $\Delta = 1.5$: forgets almost everything each step; the state is "reset" to the current input

In S4 the $\Delta$ of each channel is a learned constant: every channel has a fixed time scale, the same at every position.

Hold that thought. What if $\Delta$ could depend on the token? "Reset on a newline; remember through a name."


## S4: a time-invariant SSM is a convolution

With $\bar{\mathbf{A}}, \bar{\mathbf{B}}, \mathbf{C}$ constant, unroll the recurrence from $\mathbf{h}_0 = \mathbf{0}$:

$$
\mathbf{h}_t = \sum_{s=1}^{t} \bar{\mathbf{A}}^{\,t-s}\, \bar{\mathbf{B}}\, x_s
\quad\Rightarrow\quad
y_t = \sum_{s=1}^{t} \underbrace{\mathbf{C}\,\bar{\mathbf{A}}^{\,t-s}\,\bar{\mathbf{B}}}_{\bar K_{t-s}}\; x_s
= (\bar{\mathbf{K}} * x)_t,
$$

$$
\bar{\mathbf{K}} = \left(\mathbf{C}\bar{\mathbf{B}},\; \mathbf{C}\bar{\mathbf{A}}\bar{\mathbf{B}},\; \mathbf{C}\bar{\mathbf{A}}^2\bar{\mathbf{B}},\; \ldots,\; \mathbf{C}\bar{\mathbf{A}}^{T-1}\bar{\mathbf{B}}\right).
$$

- The output is a **causal convolution** of the input with a kernel $\bar{\mathbf{K}}$ as long as the sequence, whose shape is set by $N$ numbers. Training: compute $\bar{\mathbf{K}}$ once, convolve with an FFT in $O(T \log T)$, fully parallel. Inference: run the recurrence, $O(N)$ per token. **Two views of one model.**
- The catch is computing $\bar{\mathbf{K}}$: powers of a dense $N \times N$ matrix are $O(N^2 T)$. S4 (Gu, Goel & Ré, 2022) restricts $\mathbf{A}$ to "diagonal plus low rank" with the HiPPO initialization, which makes the kernel cheap; S4D (2022) shows a plain **diagonal** $\mathbf{A}$ works nearly as well, and that is what everything since uses.
- S4 solved benchmarks with 16,000-step dependencies (Long Range Arena, raw audio) that transformers and LSTMs could not touch. On language it lagged.



## The limit of time-invariance: selective copying

A time-invariant SSM applies **the same kernel at every position**: how much it keeps of token $s$ depends only on $t - s$, never on what token $s$ *was*.

Gu & Dao (2023), section 3.1, make this concrete with two synthetic tasks:

- **Selective copying**: remember the coloured tokens, ignore the white filler between them, reproduce the coloured ones at the end. Spacing is random.
- **Induction heads**: having seen "Harry Potter" earlier, after "Harry" predict "Potter". The associative recall behind in-context learning.


An LTI model cannot do either: it has no way to say "this token matters, keep it; that one is filler, drop it", because its dynamics do not read the content.

Linear attention (kernel $\equiv 1$) and S4 (fixed decay) both fail. Softmax attention passes trivially: its weights are a function of content.

**The fix is the RNN's gate, imported into the SSM**: let $\Delta$, and $\mathbf{B}$ and $\mathbf{C}$, be functions of $\mathbf{x}_t$.



## Mamba: the selective SSM

Mamba keeps the diagonal $\mathbf{A}$ (learned, fixed) and makes everything else a function of the input, with $\mathbf{x}_t \in \mathbb{R}^{D}$ and $N$ states per channel:

$$
\begin{aligned}
\mathbf{B}_t &= \mathrm{Linear}_N[\mathbf{x}_t], \qquad \mathbf{C}_t = \mathrm{Linear}_N[\mathbf{x}_t], \\
\Delta_t &= \mathrm{softplus}\!\left[\boldsymbol{\delta} + \mathrm{Linear}_1[\mathbf{x}_t]\right],\\
\bar{\mathbf{A}}_t &= \exp(\Delta_t \mathbf{A}), \qquad \bar{\mathbf{B}}_t = \Delta_t \mathbf{B}_t, \\
\mathbf{h}_t &= \bar{\mathbf{A}}_t \mathbf{h}_{t-1} + \bar{\mathbf{B}}_t\, \mathbf{x}_t, \qquad \mathbf{y}_t = \mathbf{C}_t \mathbf{h}_t .
\end{aligned}
$$

($\Delta_t$ is one scalar per channel; $N = 16$ in the paper. The reference code uses Euler for $\mathbf{B}$, $\bar{\mathbf{B}}_t = \Delta_t \mathbf{B}_t$; the paper's eq. (4) and Theorem 1 use ZOH for both $\mathbf{A}$ and $\mathbf{B}$, and the gated-RNN reduction on the next slide holds under ZOH.)


- $\Delta_t$ **large**: $\bar{\mathbf{A}}_t \to 0$, the state is reset and filled with $\mathbf{x}_t$. **Small**: $\bar{\mathbf{A}}_t \to \mathbf{I}$, the token is ignored. This is a forget gate, one per channel, computed from the token.
- $\mathbf{B}_t$ decides *what* about $\mathbf{x}_t$ to write; $\mathbf{C}_t$ decides what to read. Compare: keys and queries.
- The price: $\bar{\mathbf{K}}$ now differs at every position, so **the convolution view is gone**. Training has to run the recurrence, and the recurrence is sequential.
- Mamba's contribution is as much the kernel as the model: a scan fused into GPU SRAM so that the recurrence runs at the speed of a FlashAttention-style attention. The algorithm is on the next slides.


## A selective SSM is a gated RNN (derivation)

**Claim** (Gu & Dao 2023, Theorem 1). With $N = 1$, $A = -1$, $B = 1$, $C = 1$ and $\Delta_t = \mathrm{softplus}[z_t]$ where $z_t = \mathrm{Linear}[\mathbf{x}_t]$, the selective SSM is

$$
h_t = (1 - g_t)\, h_{t-1} + g_t\, x_t,
\qquad g_t = \sigma(z_t).
$$

**Proof.** Zero-order hold for both $A$ and $B$ (the paper's eq. (4) and Theorem 1; the reference code uses Euler for $B$, $\bar{B}_t = \Delta_t B_t$, and the reduction holds under ZOH), with $A=-1$, $B = 1$:
$$
\bar a_t = e^{-\Delta_t},
\qquad
\bar b_t = \frac{e^{-\Delta_t} - 1}{-1} = 1 - e^{-\Delta_t}.
$$
Now $\Delta_t = \mathrm{softplus}[z_t] = \log(1 + e^{z_t})$, so
$$
e^{-\Delta_t} = \frac{1}{1 + e^{z_t}} = 1 - \sigma(z_t) = 1 - g_t,
$$
and $\bar b_t = g_t$. Substitute: $h_t = (1-g_t)h_{t-1} + g_t x_t$. $\square$


Why it matters:

- The "discretization step size" is a **gate** in disguise; softplus plus exponential equals sigmoid. The ODE story and the LSTM story are the same story.
- The update is the GRU's $h_t = (1 - z_t) h_{t-1} + z_t \tilde h_t$ with the candidate replaced by the raw input, which is the price of keeping the recurrence linear in $h$.
- Quiz-style check: with $z_t = 0$, $g_t = \tfrac12$ and $\bar a_t = e^{-\log 2} = \tfrac12$. Consistent.


The theorem checked numerically. One channel, a random 200-step input, and a
toy selection projection $z_t = 2x_t - 1$ standing in for
$\mathrm{Linear}[\mathbf{x}_t]$. The selective SSM with ZOH for both
$\bar a_t$ and $\bar b_t$ and the gated RNN produce the same states to machine
precision. With Euler for $\mathbf{B}$, as in the reference code, the model is
still a selective SSM, but no longer this gated RNN: $\bar b_t = \Delta_t$ is
not bounded by 1.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
T = 200
x = rng.normal(size=T)                 # one channel of input
z = 2.0 * x - 1.0                      # z_t = Linear[x_t], a toy selection projection
A, B = -1.0, 1.0

Delta = np.log1p(np.exp(z))            # softplus: one step size per token
a_bar = np.exp(Delta * A)                                      # ZOH
b_bar_zoh = (np.exp(Delta * A) - 1) / (Delta * A) * Delta * B  # ZOH
b_bar_euler = Delta * B                                        # Euler, as in the reference code
g = 1 / (1 + np.exp(-z))               # sigmoid gate

def run(a, b):
    h, out = 0.0, []
    for t in range(T):
        h = a[t] * h + b[t] * x[t]
        out.append(h)
    return np.array(out)

h_gated = run(1 - g, g)
print(f"max |a_bar - (1 - g)|    = {np.abs(a_bar - (1 - g)).max():.1e}")
print(f"max |b_bar - g|          = {np.abs(b_bar_zoh - g).max():.1e}")
print(f"max |h_SSM - h_gated|    = {np.abs(run(a_bar, b_bar_zoh) - h_gated).max():.1e}   (ZOH for B)")
print(f"max |h_SSM - h_gated|    = {np.abs(run(a_bar, b_bar_euler) - h_gated).max():.2f}      (Euler for B)")

print(f"\n{'x_t':>6} {'z_t':>6} {'Delta_t':>8} {'a_bar_t':>8} {'g_t':>6}")
for t in range(5):
    print(f"{x[t]:6.2f} {z[t]:6.2f} {Delta[t]:8.3f} {a_bar[t]:8.3f} {g[t]:6.3f}")
print(f"\nz = 0: g = {1 / (1 + np.exp(0)):.2f}, a_bar = exp(-softplus(0)) = {np.exp(-np.log1p(1)):.2f}")

## The Mamba block

![The Mamba block, redrawn from Figure 3 of Gu & Dao (2023).](figs/09-Sequence-Models/mamba-block.png)


- One block replaces *both* the attention and the MLP of a transformer layer: expand the width by $E = 2$, run the selective SSM on one branch, gate it with the other, project back. Stacked with residual connections and RMSNorm (lecture 08) exactly like transformer blocks.
- The short causal **convolution** (width 4) before the SSM gives each position a glimpse of its three predecessors, which the per-channel SSM cannot do by itself.
- State per layer: $E D \times N$ numbers. For $D = 2{,}048$, $N = 16$: $65{,}536$ values, a few hundred KB. That is the whole memory of the past.
- Mamba-3B matched transformers of twice its size on language modelling at the time and ran about 5 times faster at generation (paper's claim, 2023).



## Training a selective SSM: the parallel scan (derivation)

We need all of $h_1, \ldots, h_T$ for $h_t = a_t h_{t-1} + b_t$ (one channel; $a_t = \bar a_t$, $b_t = \bar b_t x_t$), without $T$ sequential steps.

**Step 1: each step is a function.** Step $t$ maps $h \mapsto a_t h + b_t$; represent it by the pair $(a_t, b_t)$.

**Step 2: composing two steps is another such function.** Apply $(a_1, b_1)$ then $(a_2, b_2)$:
$$
a_2(a_1 h + b_1) + b_2 = (a_2 a_1)\, h + (a_2 b_1 + b_2)
\qquad\Rightarrow\qquad
(a_1, b_1) \bullet (a_2, b_2) = (a_1 a_2,\; a_2 b_1 + b_2).
$$

**Step 3: the operator is associative**, because composition of functions is. (Check: $((a_1,b_1)\bullet(a_2,b_2))\bullet(a_3,b_3) = (a_1a_2a_3,\; a_3a_2b_1 + a_3b_2 + b_3)$, and the other bracketing gives the same.) It is *not* commutative.


**Step 4: a prefix scan.** $h_t$ is the prefix product $(a_1,b_1)\bullet\cdots\bullet(a_t,b_t)$ applied to $h_0$. Computing all prefixes of an associative operator is the **parallel scan** (Blelloch, 1990): $O(T)$ total work, $O(\log T)$ sequential depth. With a diagonal $\mathbf{A}_t$ the vector case is this scalar case run per state dimension.



## Check it with numbers

$a = (0.5,\; 0.5,\; 2,\; 1)$, $b = (1, 1, 1, 1)$, $h_0 = 0$.

**Sequentially** (4 dependent steps):

$$
\begin{aligned}
h_1 &= 0.5 \cdot 0 + 1 = 1 \\
h_2 &= 0.5 \cdot 1 + 1 = 1.5 \\
h_3 &= 2 \cdot 1.5 + 1 = 4 \\
h_4 &= 1 \cdot 4 + 1 = 5
\end{aligned}
$$


**By the scan** (2 levels, each level in parallel):

Level 1, two pairs at once:
$$
\begin{aligned}
(a_1,b_1)\bullet(a_2,b_2) &= (0.5\cdot 0.5,\; 0.5\cdot 1 + 1) \\ &= (0.25,\; 1.5) \\
(a_3,b_3)\bullet(a_4,b_4) &= (2 \cdot 1,\; 1 \cdot 1 + 1) \\ &= (2,\; 2)
\end{aligned}
$$
Level 2, one combine:
$$
\begin{aligned}
(0.25, 1.5) \bullet (2, 2) &= (0.25 \cdot 2,\; 2 \cdot 1.5 + 2) \\ &= (0.5,\; 5).
\end{aligned}
$$
So $h_4 = 0.5 \cdot h_0 + 5 = 5$ and $h_2 = 0.25 \cdot h_0 + 1.5 = 1.5$. Both match.


For $T = 4$ the saving is nothing; for $T = 4{,}096$ it is 12 levels instead of 4,096 steps. Note what made it work: the *linearity* of the step in $h$. Put a $\tanh$ around $a_t h + b_t$ and the composition of two steps is no longer a step of the same form; there is no pair to combine. That is why the modern recurrences gave up the RNN's nonlinearity.


The scan in code. `parallel_scan` is the simplest log-depth scan (Hillis and
Steele's recursive doubling): at each level every element is combined with the
one `shift` places before it, and all combines in a level are independent, so
on parallel hardware a level is one step. It does $O(T \log T)$ work rather than
the $O(T)$ of Blelloch's two-sweep version drawn on the next slide, but the
depth is the same $\lceil \log_2 T \rceil$. The cell reproduces the four-step
example above, then checks a 4,096-step random recurrence with $h_0 \ne 0$
against the sequential loop.

In [ ]:
import numpy as np

def combine(p, q):
    """(a1, b1) . (a2, b2) = (a1 a2, a2 b1 + b2): apply step p, then step q."""
    a1, b1 = p
    a2, b2 = q
    return a1 * a2, a2 * b1 + b2

def sequential(a, b, h0=0.0):
    h, out = h0, []
    for a_t, b_t in zip(a, b):
        h = a_t * h + b_t
        out.append(h)
    return np.array(out)

def parallel_scan(a, b):
    """All prefixes (a_1...a_t, b-part) of the steps, in ceil(log2 T) levels."""
    a, b = np.array(a, float), np.array(b, float)
    shift, levels = 1, 0
    while shift < len(a):
        a[shift:], b[shift:] = combine((a[:-shift], b[:-shift]), (a[shift:], b[shift:]))
        shift, levels = 2 * shift, levels + 1
    return a, b, levels

a, b = [0.5, 0.5, 2, 1], [1, 1, 1, 1]
A_pre, B_pre, levels = parallel_scan(a, b)
print("prefixes:", ", ".join(f"({p:g}, {q:g})" for p, q in zip(A_pre, B_pre)), f"  [{levels} levels]")
print("scan:      h =", A_pre * 0.0 + B_pre)
print("sequential h =", sequential(a, b))

rng = np.random.default_rng(0)
T, h0 = 4096, 0.7
a = rng.uniform(0.8, 1.0, size=T)          # decays, as a_bar_t would be
b = rng.normal(size=T)                     # b_bar_t * x_t
A_pre, B_pre, levels = parallel_scan(a, b)
err = np.abs(A_pre * h0 + B_pre - sequential(a, b, h0)).max()
print(f"\nT = {T:,}: {levels} levels instead of {T:,} steps; max |h_scan - h_seq| = {err:.1e}")

## The recurrent and scan views of one computation

![Left: the recurrence, one step after another. Right: the up-sweep of a parallel scan over the same $(a_t, b_t)$, combining pairs at each level; a second sweep recovers every prefix.](figs/09-Sequence-Models/recurrent-vs-scan.png)

- Same inputs, same outputs, same $O(T)$ arithmetic. The difference is the **critical path**: $T$ on the left, $\lceil \log_2 T \rceil$ (times a small constant for the down-sweep) on the right.
- At inference you still run the left side, one token at a time, and that is the point: $O(1)$ per token, one state in memory.

## Mamba-2: state-space duality

Dao & Gu (2024) simplify Mamba's $\mathbf{A}_t$ from a diagonal to a **single scalar per head**, $\mathbf{A}_t = a_t \mathbf{I}$, with $N = 128$ states and head dimension $P = 64$. Then unroll the recurrence:

$$
\mathbf{y}_t = \sum_{s \le t} \mathbf{C}_t^{\mathsf T}\mathbf{B}_s \left(\prod_{j=s+1}^{t} a_j\right) \mathbf{x}_s
\qquad\Longrightarrow\qquad
\mathbf{y} = \mathbf{M}\,\mathbf{x},
\quad
\mathbf{M} = \mathbf{L} \circ \left(\mathbf{C}\mathbf{B}^{\mathsf T}\right),
\quad
L_{ts} = \begin{cases} \prod_{j=s+1}^{t} a_j & s \le t \\ 0 & s > t. \end{cases}
$$

![What each mixer lets step $t$ read from step $s$.](figs/09-Sequence-Models/ssd-decay-mask.png)



## SSD: read the equation both ways

**As attention.** $\mathbf{M} = \mathbf{L} \circ (\mathbf{C}\mathbf{B}^{\mathsf T})$ is causal linear attention with $\mathbf{C} \leftrightarrow \mathbf{Q}$, $\mathbf{B} \leftrightarrow \mathbf{K}$, $\mathbf{x} \leftrightarrow \mathbf{V}$, and a **decay mask** $\mathbf{L}$ in place of the all-ones causal mask. The whole sequence is a few matrix multiplies: parallel training, tensor cores, no scan needed. Cost $O(T^2)$.

**As a recurrence.** The same $\mathbf{M}$ is produced by $\mathbf{h}_t = a_t \mathbf{h}_{t-1} + \mathbf{B}_t \mathbf{x}_t^{\mathsf T}$, $\mathbf{y}_t = \mathbf{h}_t^{\mathsf T}\mathbf{C}_t$: $O(1)$ per token at inference.


**The chunked algorithm** uses both. Split $T$ into chunks of $Q$ (64 to 256) tokens:

1. Inside each chunk, use the quadratic form: a $Q \times Q$ block of $\mathbf{M}$, a matrix multiply.
2. Between chunks, pass one state $\mathbf{h}$ forward with the recurrence: $T/Q$ sequential steps instead of $T$.

Work is $O(T Q N)$, nearly all of it in matrix multiplies, which is why Mamba-2 trains 2 to 8 times faster than Mamba-1's scan at the same quality (paper's measurement). The same chunking is how Gated DeltaNet and linear-attention kernels are implemented in practice.


So "SSM" and "linear attention" are, in 2024 and after, two names for one object: a linear recurrence with a structured mask. The remaining design freedom is **what $\mathbf{A}_t$ does to the state**.


## Gated DeltaNet: the other family in 2026 models

Mamba-2's $a_t$ shrinks the **whole** state uniformly: forgetting is global. The **delta rule** (Schlag et al., 2021; Yang et al., 2024) instead overwrites the value stored under one key. In the matrix-state notation of linear attention, with $\|\mathbf{k}_t\| = 1$:

$$
\begin{aligned}
\mathbf{S}_t &= \mathbf{S}_{t-1} - \beta_t\left(\mathbf{S}_{t-1}\mathbf{k}_t - \mathbf{v}_t\right)\mathbf{k}_t^{\mathsf T} \\
&= \mathbf{S}_{t-1}\left(\mathbf{I} - \beta_t \mathbf{k}_t\mathbf{k}_t^{\mathsf T}\right) + \beta_t\, \mathbf{v}_t \mathbf{k}_t^{\mathsf T}.
\end{aligned}
$$

This is **one step of gradient descent** with learning rate $\beta_t \in (0, 1)$ on the loss $\tfrac12\|\mathbf{S}\mathbf{k}_t - \mathbf{v}_t\|^2$: "make the memory return $\mathbf{v}_t$ when queried with $\mathbf{k}_t$". Reading: $\mathbf{S}_t \mathbf{k}_t = (1 - \beta_t)\mathbf{S}_{t-1}\mathbf{k}_t + \beta_t \mathbf{v}_t$, an interpolation between the old and the new value *for that key only*.

**Gated DeltaNet** (Yang, Kautz & Hatamizadeh, 2024) adds Mamba-2's global decay $\alpha_t \in (0,1)$ in front:
$$
\mathbf{S}_t = \mathbf{S}_{t-1}\,\alpha_t\left(\mathbf{I} - \beta_t \mathbf{k}_t\mathbf{k}_t^{\mathsf T}\right) + \beta_t\, \mathbf{v}_t \mathbf{k}_t^{\mathsf T},
\qquad
\mathbf{y}_t = \mathbf{S}_t\,\mathbf{q}_t .
$$


- In the general form, $\mathbf{A}_t = \alpha_t(\mathbf{I} - \beta_t\mathbf{k}_t\mathbf{k}_t^{\mathsf T})$: identity minus a rank-one piece, no longer diagonal. The chunked parallel form still exists (Yang et al. 2024 derive it) but needs more care than a scalar decay.
- Two gates, two jobs: $\alpha_t$ clears the memory ("new document"), $\beta_t$ edits one entry ("update what *Harry* maps to").
- On the Mamba synthetic tasks and on in-context retrieval, Gated DeltaNet beats Mamba-2 at equal size, which is why it is the recurrence in the Qwen3-Next and Qwen 3.5/3.6 families (next section).



# Why 2026 models alternate layers

If the linear recurrence trains in parallel and runs in constant memory, why
has it not simply replaced attention? Because it cannot do one thing attention
does trivially: look up an exact fact from arbitrarily far back. Production
models from 2024 on resolve this by mixing the two kinds of layer. This section
gives the evidence for the trade-off and the arithmetic that makes the mix
worth it.


## Hybrid architectures: a few attention layers, many recurrent ones

![One repeating unit of four architectures. Orange layers keep a KV cache that grows with $T$; blue ones keep a fixed-size state.](figs/09-Sequence-Models/hybrid-layer-stack.png)


## Who ships what (2024 to 2026)

| Model | Recurrent layer | Attention share | Notes |
|---|---|---|---|
| Jamba (AI21, Mar 2024) | Mamba | 1 in 8 | first production hybrid; supports 256K context; up to 140K tokens fit on a single 80 GB GPU in 8-bit (model card) |
| MiniMax-01 (Jan 2025) | lightning (linear) attention | 1 in 8 | 456B total / 46B active; 1M-token context |
| Nemotron-H (NVIDIA, Apr 2025) | Mamba-2 | 4 of 52 layers (about 8%) in the 8B | 8B and 56B; up to 3x faster inference (paper) |
| Qwen3-Next (Sep 2025) | Gated DeltaNet | 1 in 4 | 80B total / 3B active; template for Qwen 3.5 and 3.6 |
| Nemotron 3 (NVIDIA, Dec 2025) | Mamba-2 | 6 of 52 layers | Nano 30B-A3B hybrid MoE: 23 Mamba-2 + 23 MoE + 6 attention layers |
| Qwen 3.5 / 3.6 (2026) | Gated DeltaNet | 1 in 4 | open weights; the hybrid is the default in this family |

Layer counts: Nemotron-H-8B has 4 self-attention layers out of 52 ([arXiv:2504.03624](https://arxiv.org/abs/2504.03624), section 2.1). Nemotron 3 Nano 30B-A3B has 52 layers, 23 Mamba-2 + 23 MoE + 6 attention (GQA, 2 KV heads), 3.5B active / 30B total ([arXiv:2512.20848](https://arxiv.org/abs/2512.20848)). Qwen3.6-27B has 64 layers, 48 Gated DeltaNet + 16 full attention (one in every four); Qwen3.6-35B-A3B has 40 = 30 + 10, the same 3:1 ratio as Qwen3-Next and Qwen3.5 (Hugging Face configs, Oct 2026).

Three observations:

- Nobody ships **pure** recurrence at scale for general language modelling. Everyone ships a **minority** of attention layers.
- The ratio ($1{:}3$ to $1{:}11$) was found by ablation, not theory. The papers report sweeps.
- Meanwhile DeepSeek V3, Llama 4 and most frontier closed models remain all-attention, with the KV cache attacked by other means (GQA, MLA; lectures 11 and 15). Both camps coexist in 2026.



## What attention still does better: associative recall

**Zoology** (Arora et al., 2023) trained matched models with attention and with gated convolutions / recurrences and asked *where* the perplexity gap comes from. Answer: almost all of it (82% in their measurement) is on tokens that require **recalling something seen earlier in the context**, such as "Hakuna Matata means no worries. Hakuna Matata means ..." They formalized it as **multi-query associative recall** (MQAR): store many key-value pairs, answer queries about any of them.

**Based** (Arora et al., 2024): for recurrent models, recall accuracy is bounded by the **size of the state**. Plot recall against state size and every linear-recurrence architecture lies on one trade-off curve; attention sits off the curve because its "state" is the whole context.


**Copying** (Jelassi et al., 2024): a transformer trained to copy strings of length 50 copies strings of length 1,000; state-space models fail to copy beyond what their state can hold, and pretrained Mamba models trail pretrained transformers on retrieval from context.

Why, in one sentence: a $d_v \times d_k$ matrix can store about $d_k$ key-value pairs without interference; a KV cache stores $T$ of them exactly. The recurrence is a **lossy compression** of the context; attention is lossless. Most tokens do not need lossless; some do, and those are the ones people notice (a name, a number, a line of code to reproduce).



## Evidence at scale: the 8B study

Waleffe et al. (NVIDIA, 2024) trained 8B-parameter Mamba, Mamba-2, Mamba-2-Hybrid and transformer models on the **same 3.5T tokens**, the cleanest comparison available:

- Pure Mamba and Mamba-2 match or beat the transformer on most standard benchmarks but **lag on MMLU** (five-shot, format-heavy) and on **in-context retrieval** tasks such as "phonebook" lookup.
- The hybrid, with only **4 attention layers out of 56** (24 Mamba-2, 4 attention, 28 MLP), closes those gaps and beats the transformer on average across the 12 standard tasks, by a couple of points.
- At inference, the hybrid generates tokens up to about **8 times faster** than the transformer at long context, because the KV cache is 1/8 the size and the recurrent layers cost $O(1)$ per token.
- The same pattern repeats in every hybrid paper since: a small number of attention layers buys back recall; the recurrent layers buy speed and memory.



## The cost picture: KV cache versus state

![Inference memory for one sequence as context grows: a worked example, not a measurement (assumptions in the text).](figs/09-Sequence-Models/kv-cache-vs-state-memory.png)


**KV cache**, Llama 3 8B shape (32 layers, 8 KV heads, $d_{\text{head}} = 128$, bf16): **128 KiB per token** (derived in lecture 11).

At 128K tokens: **16 GiB** for *one* sequence. At 1M: 122 GiB, more than the GPU.

**Mamba-2 state**, 64 layers, 128 heads $\times$ 64 $\times$ $N=128$, bf16: 2 MiB per layer, **128 MiB** total, for any $T$. Equal to the KV cache of a 1,000-token context.

**Hybrid** (1 in 4 layers attention): a quarter of the slope. Serving 4 times as many concurrent long conversations per GPU is the business case.


Memory is the binding constraint on an inference server (lecture 15): the KV cache, not the weights, decides how many users share a GPU. Decode cost per token has the same shape: $O(T)$ reads for attention, $O(1)$ for the recurrence.


The numbers on this slide, as a calculator. Cache and state sizes are in binary
units (KiB, MiB, GiB), the GPU in decimal GB. The hybrid has the same 32 layers
as the attention model, a quarter of them attention and the rest carrying a
Mamba-2 state. Try a Mamba-1 style state ($P = 1$ per head, `heads` $= 2D$,
$N = 16$), fewer KV heads, or a hybrid with one attention layer in eight.

In [ ]:
L, H_kv, d_head = 32, 8, 128              # attention model: Llama 3 8B shape
n_layers, heads, P, N = 64, 128, 64, 128  # Mamba-2 style model: layers, heads, head dim, state dim
attn_share = 1 / 4                        # hybrid: fraction of the L layers that are attention
bytes_per = 2                             # bf16
gpu = 80e9                                # bytes on one GPU

KiB, MiB, GiB = 2**10, 2**20, 2**30
kv_per_token = 2 * L * H_kv * d_head * bytes_per    # K and V, every layer
state_per_layer = heads * P * N * bytes_per
ssm_state = n_layers * state_per_layer
n_attn = round(attn_share * L)
print(f"KV cache: {kv_per_token / KiB:.0f} KiB per token")
print(f"SSM state: {state_per_layer / MiB:.0f} MiB per layer, {ssm_state / MiB:.0f} MiB in all, for any T")
print(f"the SSM state equals the KV cache of {ssm_state / kv_per_token:,.0f} tokens")

print(f"\n{'context T':>10} {'attention':>10} {'hybrid':>8} {'pure SSM':>9}   (GiB per sequence)")
for T in [1_000, 8_192, 131_072, 1_000_000]:
    attn = T * kv_per_token
    hybrid = T * kv_per_token * n_attn / L + (L - n_attn) * state_per_layer
    print(f"{T:>10,} {attn / GiB:>10.3f} {hybrid / GiB:>8.3f} {ssm_state / GiB:>9.3f}")
print(f"\ncontext whose cache alone fills {gpu / 1e9:.0f} GB: "
      f"attention {gpu / kv_per_token:,.0f} tokens, hybrid {gpu * L / (n_attn * kv_per_token):,.0f}")

E, D, N1 = 2, 2048, 16                    # the Mamba-1 block slide: expansion, width, states
print(f"\nMamba-1 block state, D = {D}: E*D*N = {E * D * N1:,} values per layer "
      f"= {E * D * N1 * 4 / KiB:.0f} KiB in fp32")

## Honest limits

- **Recall is bought, not solved.** A hybrid's exact-retrieval ability lives in its attention layers; the recurrent layers remain a lossy summary. Needle-in-a-haystack tests at 1M tokens are mostly tests of the attention layers.
- **The quality evidence is strongest below about 10B parameters** and on 2024-25 training recipes. The largest open all-attention models (DeepSeek V3.x, Llama 4) have not been beaten by a hybrid of equal size on reasoning benchmarks in a controlled comparison, because no such comparison exists at that scale.
- **The ratios are empirical.** 1:3, 1:7, 1:11: chosen by sweeps on a given budget. No theory predicts the right mix, and at least one lab has gone back to full attention for a later model after shipping a hybrid, citing evaluation and infrastructure immaturity: MiniMax-M2 (230B total / 10B active) returned to full softmax attention in all 62 layers, after MiniMax-Text-01's 7:1 lightning:softmax mix (HF configs).
- **Speed is a kernel property.** Mamba, SSD and Gated DeltaNet are fast because of fused GPU kernels (the Mamba scan, the chunked SSD algorithm, the `flash-linear-attention` library). A plain PyTorch loop over $t$ is slower than attention. Infrastructure (serving frameworks, quantization, speculative decoding) lagged attention by a year or more.
- **The ODE story is partly decoration.** Mamba-2 and Gated DeltaNet drop the continuous-time framing; $\Delta$ is a gate and $\mathbf{A}$ is a decay. The useful residue of S4 is the diagonal, bounded $\mathbf{A}$ and the initialization, not the differential equation.
- **Length generalization is not automatic** for recurrences either: a decay rate tuned on 8K-token training sequences may be wrong at 256K.


# Summary

What to take away from this lecture; the recap deck and the knowledge check
are built from this list. Lecture 10 takes up attention and the transformer
in full; lecture 11 covers grouped-query attention and mixture of experts;
lecture 15 returns to the KV cache from the serving side.


## Summary

- A sequence model is a model of $\mathrm{Pr}(\mathbf{x}_t \mid \mathbf{x}_{<t})$; the chain rule makes next-element prediction a model of the whole sequence. Architectures differ in **what state summarizes the past** and what that state costs.
- RNN: $\mathbf{h}_t = \mathrm{a}[\boldsymbol{\beta} + \boldsymbol{\Omega}_h \mathbf{h}_{t-1} + \boldsymbol{\Omega}_x \mathbf{x}_t]$, one cell with shared weights unrolled over time; trained by backpropagation through time, which is ordinary backprop on the unrolled graph with gradients summed over the copies.
- The long-range gradient $\partial \mathbf{h}_t / \partial \mathbf{h}_k = \prod_{j} \mathrm{diag}[\mathrm{a}'[\mathbf{f}_j]]\,\boldsymbol{\Omega}_h$ is bounded by $(\gamma \sigma_{\max})^{t-k}$: it vanishes if $\sigma_{\max} < 1/\gamma$ and can explode if $\sigma_{\max} > 1/\gamma$. Clipping handles explosion; gates (LSTM, GRU; module M1) give the state an additive path with a learned forget factor.
- Attention won because its path length is $O(1)$ (no Jacobian product) and its training is $O(1)$ sequential steps; it pays with $O(T^2)$ compute and a KV cache that grows linearly with $T$.
- Dropping the softmax makes attention a recurrence with a matrix state, $\mathbf{S}_t = \mathbf{S}_{t-1} + \mathbf{v}_t\mathbf{k}_t^{\mathsf T}$; every modern recurrence is $\mathbf{h}_t = \mathbf{A}_t\mathbf{h}_{t-1} + \mathbf{B}_t\mathbf{x}_t$, $\mathbf{y}_t = \mathbf{C}_t\mathbf{h}_t$, linear in the state, with $\|\mathbf{A}_t\| \le 1$ by construction.
- State-space models discretize $\mathbf{h}' = \mathbf{A}\mathbf{h} + \mathbf{B}x$ with zero-order hold, $\bar{\mathbf{A}} = \exp(\Delta\mathbf{A})$; a time-invariant SSM (S4) is a long convolution, trainable by FFT, but cannot select by content. Mamba makes $\Delta_t, \mathbf{B}_t, \mathbf{C}_t$ functions of the input; with $N=1$ and softplus, $\Delta_t$ *is* a sigmoid gate.
- A linear recurrence is trained in parallel by a scan: steps are pairs $(a, b)$, composed by $(a_1,b_1)\bullet(a_2,b_2) = (a_1a_2, a_2b_1 + b_2)$, associative, so all prefixes take $O(\log T)$ depth. Mamba-2 (SSD) shows the scalar-decay case equals linear attention with a decay mask and trains by chunked matrix multiplies; Gated DeltaNet replaces uniform decay with a per-key delta-rule update.
- Recurrences compress the context; attention stores it. Associative recall and copying are where that difference shows (Zoology, Based, Repeat After Me). Hybrid models (Jamba, Nemotron 3, Qwen 3.5/3.6) therefore keep about one attention layer in roughly four to twelve (3:1 for Qwen3.5/3.6, 1:9 Nemotron 3 Nano, 1:13 Nemotron-H): constant-size state for most of the stack, a KV cache that grows at a fraction of the rate, and exact recall where it matters.